# Validation Framework Demo

Demonstrates the full validation pipeline on **synthetic data**, showing:
1. Scoring with multiple HRD signatures
2. Comprehensive metrics computation
3. Head-to-head comparison across methods
4. Reversion analysis
5. Report generation

When real models and data are available, replace the synthetic data loading with actual cohort validators.

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_style('whitegrid')
sns.set_context('notebook')

from v2.validation.scoring import SignatureScorer, CentroidScorer
from v2.validation.metrics import ValidationMetrics
from v2.validation.head_to_head import HeadToHead
from v2.validation.reversion_analysis import ReversionAnalysis
from v2.validation.validation_report import ValidationReport

## 1. Create Synthetic Validation Cohort

We simulate a cohort with:
- 200 samples (100 HRD, 100 HRP)
- Expression for key HR-pathway genes with realistic separation
- Binary labels + platinum response + survival data

In [ ]:
n_samples = 200
n_hrd = 100
n_hrp = 100

# Key genes for HRD signatures
hrd_genes = ['BRCA1', 'BRCA2', 'RAD51', 'PALB2', 'RAD51C', 'POLQ',
             'PARP1', 'CHEK1', 'ATR', 'EXO1', 'FANCI', 'FANCD2',
             'STAT1', 'CXCL10', 'CCL5', 'IDO1', 'MAPT', 'MYB',
             'ESR1', 'PGR', 'GATA3', 'FOXA1', 'MKI67', 'TOP2A',
             'CCNA2', 'CCNB1', 'MCM2', 'CDK1', 'BUB1', 'AURKB']

# Generate expression: HRD samples have distinct pattern
# HR genes DOWN in HRD, immune genes UP, proliferation UP
expr_data = np.random.randn(n_samples, len(hrd_genes))

# HR repair genes: lower in HRD
hr_idx = [hrd_genes.index(g) for g in ['BRCA1', 'BRCA2', 'RAD51', 'PALB2', 'RAD51C']]
expr_data[:n_hrd, hr_idx] -= 1.5

# POLQ: higher in HRD (compensatory alt-EJ)
polq_idx = hrd_genes.index('POLQ')
expr_data[:n_hrd, polq_idx] += 2.0

# Immune genes: higher in HRD (cGAS-STING activation)
immune_idx = [hrd_genes.index(g) for g in ['STAT1', 'CXCL10', 'CCL5', 'IDO1']]
expr_data[:n_hrd, immune_idx] += 1.2

# ER genes: lower in HRD (BRCA1 tumors tend to be ER-negative)
er_idx = [hrd_genes.index(g) for g in ['ESR1', 'PGR', 'GATA3', 'FOXA1']]
expr_data[:n_hrd, er_idx] -= 1.8

# Proliferation: higher in HRD
prolif_idx = [hrd_genes.index(g) for g in ['MKI67', 'TOP2A', 'CCNA2', 'CDK1']]
expr_data[:n_hrd, prolif_idx] += 1.0

sample_ids = [f'SAMPLE_{i:03d}' for i in range(n_samples)]
expression_df = pd.DataFrame(expr_data, index=sample_ids, columns=hrd_genes)

print(f'Expression matrix: {expression_df.shape}')
expression_df.head()

In [ ]:
# Create labels
labels = np.array([1]*n_hrd + [0]*n_hrp)

# Platinum response: correlated with HRD but noisy
plat_response = labels.copy().astype(float)
# Flip ~15% of labels for noise
flip_idx = np.random.choice(n_samples, size=30, replace=False)
plat_response[flip_idx] = 1 - plat_response[flip_idx]

# Survival: HRD has better PFI under platinum
pfi_time = np.where(labels == 1,
                    np.random.exponential(24, n_samples),  # HRD: longer PFI
                    np.random.exponential(10, n_samples))  # HRP: shorter PFI
pfi_event = np.random.binomial(1, 0.7, n_samples)

# BRCA status for reversion analysis
brca_status = (['BRCA1']*30 + ['BRCA2']*20 + ['HRD_other']*50 + ['wildtype']*100)

# Scar scores: always high in HRD (even with reversion)
hrd_sum = np.where(labels == 1,
                   np.random.normal(55, 10, n_samples),
                   np.random.normal(15, 8, n_samples))
hrd_sum = np.clip(hrd_sum, 0, 100)

labels_df = pd.DataFrame({
    'label': labels,
    'platinum_response': plat_response.astype(int),
    'pfi_time': pfi_time,
    'pfi_event': pfi_event,
    'BRCA_status': brca_status,
    'HRD_sum': hrd_sum,
}, index=sample_ids)

print(f'Labels: {labels_df.shape}')
print(f'HRD: {(labels==1).sum()}, HRP: {(labels==0).sum()}')
labels_df.head()

## 2. Scoring with Multiple Signatures

In [ ]:
# Create scorers for different methods
scorers = {
    'Severson': SignatureScorer(method='severson'),
    'PARPi7': SignatureScorer(method='parpi7'),
    'POLQ': SignatureScorer(method='polq'),
    'Peng': SignatureScorer(method='peng'),
}

# Score each
for name, scorer in scorers.items():
    scores = scorer.score(expression_df)
    print(f'{name}: mean HRD={scores[:n_hrd].mean():.3f}, '
          f'mean HRP={scores[n_hrd:].mean():.3f}, '
          f'separation={scores[:n_hrd].mean() - scores[n_hrd:].mean():.3f}')

## 3. Comprehensive Metrics

In [ ]:
# Pick one scorer and compute all metric types
polq_scores = scorers['POLQ'].score(expression_df)

# Binary metrics
bm = ValidationMetrics.binary_metrics(
    labels_df['label'].values, polq_scores.values
)
print('=== Binary Metrics (POLQ) ===')
for k, v in bm.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

In [ ]:
# Survival metrics
sm = ValidationMetrics.survival_metrics(
    labels_df['pfi_time'].values,
    labels_df['pfi_event'].values,
    polq_scores.values,
)
print('=== Survival Metrics (POLQ) ===')
for k, v in sm.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

In [ ]:
# Drug response metrics
dm = ValidationMetrics.drug_response_metrics(
    labels_df['platinum_response'].values,
    polq_scores.values,
)
print('=== Drug Response Metrics (POLQ) ===')
for k, v in dm.items():
    if isinstance(v, (float, int)):
        print(f'  {k}: {v}')
    elif not isinstance(v, list):
        print(f'  {k}: {v}')

In [ ]:
# Calibration metrics
# Rescale POLQ to [0,1] for calibration
from sklearn.preprocessing import minmax_scale
polq_prob = pd.Series(minmax_scale(polq_scores), index=polq_scores.index)

cm = ValidationMetrics.calibration_metrics(
    labels_df['label'].values, polq_prob.values
)
print('=== Calibration Metrics ===')
print(f'  Brier score: {cm["brier"]:.4f}')
print(f'  Hosmer-Lemeshow p: {cm["hosmer_lemeshow_p"]:.4f}')

## 4. Head-to-Head Comparison

In [ ]:
h2h = HeadToHead(signatures=scorers)
comparison = h2h.run_all(
    expression_df, labels_df,
    label_col='label',
    response_col='platinum_response',
)

print('=== Head-to-Head Summary ===')
h2h.summary_table()

In [ ]:
# ROC comparison plot
fig = h2h.plot_roc_comparison(title='Synthetic Cohort: ROC Comparison')
plt.show()

In [ ]:
# Agreement heatmap
fig = h2h.plot_agreement_heatmap(title='Inter-signature Spearman Correlation')
plt.show()

In [ ]:
# Score distributions by label
fig = h2h.plot_score_distributions()
plt.show()

In [ ]:
# Inter-signature agreement table
scores_dict = {name: scorer.score(expression_df) for name, scorer in scorers.items()}
agreement = ValidationMetrics.signature_agreement(scores_dict)
agreement

In [ ]:
# Discordant samples
discordant = h2h.identify_discordant_samples(threshold=0.5)
print(f'Discordant samples: {len(discordant)}')
discordant.head(10)

## 5. Reversion Analysis

In [ ]:
# Add reversion flags to a subset of BRCA-mutant samples
# Simulate: 10 BRCA1-mutant samples have reversions
labels_df['has_reversion'] = False
reversion_idx = labels_df.index[10:20]  # samples 10-19 (BRCA1 mutant)
labels_df.loc[reversion_idx, 'has_reversion'] = True

# Make reversion samples look more like HRP in expression
# (simulate restored HR function)
for idx in reversion_idx:
    i = list(expression_df.index).index(idx)
    expression_df.iloc[i, hr_idx] += 1.2  # Restore HR gene expression
    expression_df.iloc[i, polq_idx] -= 1.5  # POLQ drops when HR restored

print(f'Reversion samples: {labels_df["has_reversion"].sum()}')
print(f'BRCA status distribution:\n{labels_df["BRCA_status"].value_counts()}')

In [ ]:
# Run reversion analysis
rev = ReversionAnalysis(scorer=scorers['POLQ'], scar_score_col='HRD_sum')

result = rev.simulate_reversion_effect(
    expression_df, labels_df,
    brca_col='BRCA_status',
    reversion_col='has_reversion',
)

print('=== Reversion Analysis Results ===')
print(result['interpretation'])
print()
print('=== Statistical Tests ===')
for test_name, test_vals in result['tests'].items():
    print(f'  {test_name}: p={test_vals["ttest_p"]:.4e}')

In [ ]:
# Reversion comparison plot
fig = rev.plot_reversion_comparison(result)
plt.show()

In [ ]:
# Expression vs scar scatter
fig = rev.plot_scatter_expr_vs_scar(result)
plt.show()

## 6. Report Generation

In [ ]:
# Build report
report = ValidationReport(
    model_name='softHRD_v2_demo',
    output_dir='validation_results_demo',
)

# Add cohort results (simulating what each validator would produce)
for name, scorer in scorers.items():
    scores = scorer.score(expression_df)
    metrics = {
        'n_samples': n_samples,
        'endpoint': 'synthetic_label',
        'binary_metrics': ValidationMetrics.binary_metrics(
            labels_df['label'].values, scores.values
        ),
        'drug_response_metrics': ValidationMetrics.drug_response_metrics(
            labels_df['platinum_response'].values, scores.values
        ),
    }
    report.add_cohort_result(f'Synthetic-{name}', metrics)

# Add head-to-head
report.add_head_to_head(comparison, h2h_obj=h2h)

# Add reversion
report.add_reversion_result(result)

report.add_note(
    'This report was generated on synthetic data for demonstration. '
    'Replace with real cohort validators for production use.'
)

# Generate
report_path = report.generate_report()
print(f'Report saved to: {report_path}')

In [ ]:
# Summary table
report.generate_summary_table()

## 7. What Real Validation Looks Like

Once the trained model is available (Task #6), plug it in like this:

```python
# Load trained model and feature engine
import joblib
model = joblib.load('v2/models/softhrd_v2_model.pkl')
engine = joblib.load('v2/models/feature_engine.pkl')

# Create v2 scorer
v2_scorer = SignatureScorer(
    method='softHRD_v2',
    model=model,
    feature_engine=engine,
)

# Run on real cohorts
from v2.validation.cohort_validators import (
    TCGAOVValidator, ISPY2Validator, GEOOvarianValidator,
    CellLineValidator, PanCancerValidator,
)

validators = [
    TCGAOVValidator(),
    ISPY2Validator(),
    GEOOvarianValidator(geo_id='GSE9891'),
    CellLineValidator(drug_name='Olaparib'),
    PanCancerValidator(cancer_types=['PRAD', 'PAAD']),
]

report = ValidationReport(model_name='softHRD_v2')
for validator in validators:
    result = validator.validate(v2_scorer)
    report.add_cohort_result(result['cohort_name'], result)

# Head-to-head with competing methods
all_scorers = {
    'softHRD_v2': v2_scorer,
    'Severson': SignatureScorer(method='severson'),
    'PARPi7': SignatureScorer(method='parpi7'),
    'POLQ': SignatureScorer(method='polq'),
}
h2h = HeadToHead(all_scorers)
# ... run on each cohort ...

report.generate_report()
```